# 21 · Confirming a 1h trendline break

A 1h line gives way on a 5m candle. That candle is **rs**. Before the trade is taken, a later
candle has to prove the move held:

* **rs is the bar the break STARTED on** (`TL_CONFIRM_FROM = "first"`). The break condition is a
  state, not an event, so a 1h line reads broken across a *run* of 5m bars; letting every one of
  them be an rs hands the rule the easiest reference in the run instead of the real break candle;
* within **`TL_CONFIRM_N`** candles of rs, one must close **beyond rs** — below its low for a
  short, above its high for a long — and be the trade's own colour;
* if instead a candle **closes back over rs**, the setup is cancelled (`TL_CONFIRM_CANCEL`) — a
  bounce voids it rather than leaving it armed.

Set per line timeframe, so 1h breaks must prove themselves while 15m and 5m are taken as they come.

Everything else matches notebook 20, and the run below is repeated with the confirmation OFF, so
the only difference between the two columns is this rule.

Settings: **`nb21_config.py`**.

## 1 · Settings

In [1]:
import sys, os, importlib, warnings; warnings.filterwarnings('ignore')
try:
    import quant
except ModuleNotFoundError:
    sys.path.insert(0, os.path.abspath('..')); import quant
import numpy as np, pandas as pd
from quant.viz import AS_OF_JS

for _p in (os.getcwd(), os.path.join(os.getcwd(), "notebooks"), os.path.abspath("..")):
    if os.path.isfile(os.path.join(_p, "nb21_config.py")) and _p not in sys.path:
        sys.path.insert(0, _p)
import nb21_config as CFG
importlib.reload(CFG)

print(f"{CFG.SYMBOL} {CFG.TF}  {CFG.VIEW_START} -> {CFG.AS_OF or CFG.END}")
_fit = (CFG.TF,) + tuple(t for t in CFG.LINE_TFS if t != CFG.TF)   # the chart's TF is always fitted
print(f"lines fitted on {'+'.join(_fit)}  ·  traded on "
      f"{'+'.join(getattr(CFG, 'TRADE_TFS', None) or _fit)}")
print(f"confirmation: rs = {CFG.TL_CONFIRM_FROM} break bar · {CFG.TL_CONFIRM_N}  "
      f"{'all n' if CFG.TL_CONFIRM_ALL else 'one of n'} · {CFG.TL_CONFIRM_MODE} beyond rs · "
      f"{'own colour' if CFG.TL_CONFIRM_COLOUR else 'any colour'} · "
      f"{'cancels on a close back over rs' if CFG.TL_CONFIRM_CANCEL else 'no cancel'}")
_rules = [(getattr(CFG, d, {}) or {}).get("tag", t) for u, d, t in (
    ("USE_TP_MOVE", "TP_MOVE", "MOVED"), ("USE_FVG_EXIT", "FVG_EXIT", "FVG"),
    ("USE_TP_DECAY", "TP_DECAY", "DECAY"), ("USE_STOP_STEPS", "STOP_STEPS", "STEP"),
    ("USE_STOP_TIME", "STOP_TIME", "TSTOP"), ("USE_HARD_CLOSE", "HARD_CLOSE", "HARD"))
    if getattr(CFG, u, False)]
print(f"exits: target {CFG.TP_RR:g}R  ·  rules on: {', '.join(_rules) or 'none'}")
print(f"settings from {CFG.__file__}")

BTCUSDT 5m  2026-06-01 -> 2026-06-20
lines fitted on 5m+15m+1h+4h  ·  traded on 15m+1h+4h
confirmation: rs = first break bar · {'1h': 3, '15m': 3, '5m': 2, '*': 0}  one of n · close beyond rs · own colour · cancels on a close back over rs
exits: target 3R  ·  rules on: FVG
settings from C:\Users\Talal Zahid\Desktop\Talal\Binance\notebooks\nb21_config.py


## 2 · With the confirmation, and without

`variant()` re-runs the identical study with the rule switched off. Nothing else changes, so every
difference below belongs to the confirmation.

In [2]:
from quant.research import fvg_lab, variant, compare_runs, trade_diff

on  = fvg_lab(CFG)
off = fvg_lab(variant(CFG, TL_CONFIRM_N=0, TL_CONFIRM_CANCEL=False))

compare_runs(confirmation_off=off, confirmation_on=on)

[21:16:20] INFO | quant.data | cache HIT  binance BTCUSDT 5m -> pushdown load
[21:16:20] INFO | quant.data | cache HIT  binance BTCUSDT 4h -> pushdown load
[21:16:20] INFO | quant.data | cache HIT  binance BTCUSDT 1h -> pushdown load
[21:16:20] INFO | quant.data | cache HIT  binance BTCUSDT 15m -> pushdown load
[21:16:20] INFO | quant.data | cache HIT  binance BTCUSDT 4h -> pushdown load
[21:16:20] INFO | quant.data | cache HIT  binance BTCUSDT 1h -> pushdown load
[21:16:20] INFO | quant.data | cache HIT  binance BTCUSDT 15m -> pushdown load
[21:16:20] INFO | quant.data | cache HIT  binance BTCUSDT 1m -> pushdown load
[21:16:23] INFO | quant.engine | backtest done | bars=14,401 trades=1 final_cash=9891.76 return=-1.08% dd=3.67% | 51.4 ms
[21:16:23] INFO | quant.engine | backtest done | bars=14,401 trades=4 final_cash=10736.81 return=7.37% dd=0.88% | 1.0 ms
[21:16:23] INFO | quant.data | cache HIT  binance BTCUSDT 5m -> pushdown load
[21:16:23] INFO | quant.data | cache HIT  binance BTC

,run,trades,total R,wins,R / trade
0,confirmation off,10,2.00,4,0.200
1,confirmation on,5,6.81,4,1.362


## 3 · Which trades it changed

A confirmation can only move an entry later or remove it, so read the rows that appear in one
column and not the other. `1h` in `trigger` is the timeframe this rule is set for.

In [3]:
trade_diff(off=off, on=on)

,side off,trigger off,R off,close_reason off,side on,trigger on,R on,close_reason on
entry,,,,,,,,
2026-06-01 01:40,short,15m,2.802544,take_profit,NaN,NaN,NaN,NaN
2026-06-01 01:50,NaN,NaN,NaN,NaN,short,15m,2.459663,fvg_exit
2026-06-02 07:00,short,15m,2.871383,take_profit,NaN,NaN,NaN,NaN
2026-06-02 08:45,NaN,NaN,NaN,NaN,short,15m,2.768037,take_profit
2026-06-05 18:50,short,1h,-1.053178,stop_loss,NaN,NaN,NaN,NaN
2026-06-05 22:50,long,15m,-1.135041,stop_loss,NaN,NaN,NaN,NaN
2026-06-06 08:25,long,15m,-1.140345,stop_loss,NaN,NaN,NaN,NaN
2026-06-06 21:30,long,15m,-1.209889,stop_loss,NaN,NaN,NaN,NaN
2026-06-07 22:10,long,15m,-1.055388,stop_loss,NaN,NaN,NaN,NaN


## 4 · The setups that did NOT trade

A filter can only be judged against what it removed — and the chart shows only what survived, so a
rule that deletes a good setup looks identical to one that deletes nothing. Every break that was
turned down, with the reason:

* **entry rule** — the `ENTRY_RULES` sentence was false on the break bar
* **not confirmed** — the window expired with no qualifying candle
* **cancelled** — price closed back over rs inside the window
* **already open** — the signal was fine, but a position was already running

On the chart they are one legend entry per reason, **all starting off**, so the view opens on price
and trades exactly as before. Click one in the legend to bring it in.

In [4]:
on.rejected.groupby(["tf", "reason"]).size().rename("breaks").reset_index()

,tf,reason,breaks
0,15m,already open,2
1,15m,entry rule,12
2,15m,not confirmed,5
3,1h,not confirmed,1


## 5 · Two optional exits

Both leave the entry alone and change only how a trade ends.

* **Strategy 1, tag `MOVED`** (`USE_TP_MOVE`) — price got `near` of the way to the target and a
  close then gave `back` of it back, or the trade has been open `after_bars`: the target moves to
  `to` of its distance. Judged on the close, used from the next bar.
* **Strategy 2, tag `FVG`** (`USE_FVG_EXIT`) — a live gap from `tfs` sits within `near_r` R of the
  target when the trade opens: price filling it closes the trade (at its `touch`, `ce` or `full`).

§3 of the config has four more, each off until switched on and each with its own tag:
**`DECAY`** (the target shrinks every N hours open), **`STEP`** and **`TSTOP`** (the stop moves
toward profit at a share of the way to the target, or after N hours), and **`HARD`** (out after N
hours). Hours can be set per line timeframe; a trade uses the settings of the coarsest line that
fired it. Every tag is written on the chart's exit marker — `TP·DECAY`, `SL·TSTOP`, `HARD` — beside
the plain `TP`, `SL` and `BE`.

Each arm of `COMPARE` re-runs the notebook with only its own change, so the columns differ by
exactly what they name. Both rules were **designed from one trade** in this window, which is why it
looks good here — the verdict over two years is in `docs/WHAT_WORKS.md`.

In [5]:
from quant.research import run_arms

arms = run_arms(CFG)          # one run per arm of COMPARE
arms.table()

[21:16:25] INFO | quant.data | cache HIT  binance BTCUSDT 5m -> pushdown load
[21:16:25] INFO | quant.data | cache HIT  binance BTCUSDT 4h -> pushdown load
[21:16:25] INFO | quant.data | cache HIT  binance BTCUSDT 1h -> pushdown load
[21:16:25] INFO | quant.data | cache HIT  binance BTCUSDT 15m -> pushdown load
[21:16:25] INFO | quant.data | cache HIT  binance BTCUSDT 4h -> pushdown load
[21:16:25] INFO | quant.data | cache HIT  binance BTCUSDT 1h -> pushdown load
[21:16:25] INFO | quant.data | cache HIT  binance BTCUSDT 15m -> pushdown load
[21:16:25] INFO | quant.data | cache HIT  binance BTCUSDT 1m -> pushdown load
[21:16:26] INFO | quant.engine | backtest done | bars=14,401 trades=1 final_cash=9891.76 return=-1.08% dd=3.67% | 1.3 ms
[21:16:26] INFO | quant.engine | backtest done | bars=14,401 trades=4 final_cash=10736.81 return=7.37% dd=0.88% | 0.9 ms
[21:16:26] INFO | quant.engine | backtest done | bars=14,401 trades=1 final_cash=9891.76 return=-1.08% dd=3.67% | 1.1 ms
[21:16:26] 

,run,trades,total R,wins,R / trade,forced_close_end,fvg_exit,stop_loss,take_profit,tp_moved
0,neither,5,3.70,3,0.740,1,0,2,2,0
1,move target,5,9.42,5,1.885,1,0,0,2,2
2,fvg exit,5,6.81,4,1.362,1,2,1,1,0
3,both,5,9.61,5,1.922,1,2,0,1,1


In [6]:
arms.diff()

,side,trigger,neither R,neither exit,move target R,move target exit,fvg exit R,fvg exit exit,both R,both exit
entry,,,,,,,,,,
2026-06-01 01:50,short,15m,2.84,target,2.84,target,2.46,fvg exit,2.46,fvg exit
2026-06-02 08:45,short,15m,2.77,target,2.77,target,2.77,target,2.77,target
2026-06-09 09:35,short,15m,-1.17,stop,1.75,moved target,2.31,fvg exit,2.31,fvg exit
2026-06-11 17:55,long,15m,-1.08,stop,1.72,moved target,-1.08,stop,1.72,moved target
2026-06-17 19:35,short,15m,0.35,still open,0.35,still open,0.35,still open,0.35,still open


## 6 · The chart

Drawn as the settings stand. **Every exit marker carries its tag** (`EXIT_LABELS`): `TP` target,
`SL` stop, `BE` stop at breakeven, `TP·MOVED` / `TP·FVG` for the two exit strategies. Hover a marker
for the reason in words — which trigger moved the target and from where to where, or which gap
filled. Turn a strategy on in §3 of the config to see it here; a moved target also shows as a step
in the trade's own view, an FVG exit as a second target line.

In [7]:
fig = on.figure()
# fig

In [8]:
from quant.viz import open_in_browser

open_in_browser(fig, "confirm.html", extra_js=AS_OF_JS)

WindowsPath('C:/Users/Talal Zahid/Desktop/Talal/Binance/notebooks/confirm.html')

---
### What to change

| want | setting |
|---|---|
| any bar of the break may be rs | `TL_CONFIRM_FROM = "every"` |
| more or less grace after rs | `TL_CONFIRM_N = {"1h": 3, "*": 0}` |
| every candle must hold, not just one | `TL_CONFIRM_ALL = True` |
| the whole candle beyond rs, not its close | `TL_CONFIRM_MODE = "full"` |
| a bounce should not void the setup | `TL_CONFIRM_CANCEL = False` |
| confirm 15m breaks too | add `"15m"` to `TL_CONFIRM_N` |
| stop trading one timeframe's lines (still drawn) | drop it from `TRADE_TFS`, e.g. `("15m", "1h")` |
| rename what the chart writes | `tag=` in the rule's settings |
| shrink the target with time / move the stop / close after N hours | `USE_TP_DECAY` / `USE_STOP_STEPS`, `USE_STOP_TIME` / `USE_HARD_CLOSE` (§3) |
| no labels on the exit markers | `EXIT_LABELS = False` (§8) |
| move the target after a near miss | `USE_TP_MOVE = True`, tune `TP_MOVE` |
| exit when a gap near the target fills | `USE_FVG_EXIT = True`, tune `FVG_EXIT` |
| a different A/B table | edit `COMPARE` |

Eight trades over nine days is not evidence either way. `experiments/trendline_wait_search.py`
scores this across six windows against a chance baseline; that is what settles it.